In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

**Environment & Dataset Path Verification**

In [1]:
# ============================================================
# CELL 1 — Environment & Dataset Path Verification
# ============================================================

import os
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

# Kaggle dataset paths
PLANTVILLAGE_ROOT = "/kaggle/input/datasets/mahmudnadim/robust-crop-disease-plantvillage"
SPLIT_ROOT = "/kaggle/input/datasets/mahmudnadim/robust-crop-disease-splits/splits-20260929T040001Z-1-001/splits"

print("\nPlantVillage path exists:", os.path.exists(PLANTVILLAGE_ROOT))
print("Split path exists:", os.path.exists(SPLIT_ROOT))

print("\nPlantVillage contents:")
print(os.listdir(PLANTVILLAGE_ROOT)[:10])

print("\nSplit contents:")
print(os.listdir(SPLIT_ROOT))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8

PlantVillage path exists: True
Split path exists: True

PlantVillage contents:
['color']

Split contents:
['color_test.txt', 'color_train.txt']


In [4]:
# ============================================================
# CELL 2 — Inspect Split Dataset
# ============================================================

import os

SPLIT_DATASET_ROOT = "/kaggle/input/datasets/mahmudnadim/robust-crop-disease-splits"

for root, dirs, files in os.walk(SPLIT_DATASET_ROOT):
    level = root.replace(SPLIT_DATASET_ROOT, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    
    for file in files:
        print(f"{indent}  {file}")

robust-crop-disease-splits/
  train_validation_split.npz
  splits-20260929T040001Z-1-001/
    splits/
      color_test.txt
      color_train.txt


**Build the exact dataframe and verify class distribution**

In [7]:
# ============================================================
# CELL 4 — Corrected Dataset Reconstruction
# ============================================================

import os
import re
import pandas as pd

# ------------------------------------------------------------
# Exact 38-class order
# ------------------------------------------------------------
CLASS_NAMES = [
    "Apple___Apple_scab",
    "Apple___Black_rot",
    "Apple___Cedar_apple_rust",
    "Apple___healthy",
    "Blueberry___healthy",
    "Cherry_(including_sour)___Powdery_mildew",
    "Cherry_(including_sour)___healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
    "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)",
    "Peach___Bacterial_spot",
    "Peach___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Raspberry___healthy",
    "Soybean___healthy",
    "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch",
    "Strawberry___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy"
]

CLASS_TO_IDX = {
    name: idx for idx, name in enumerate(CLASS_NAMES)
}

# ------------------------------------------------------------
# Resolve split-file paths against actual Kaggle dataset
# ------------------------------------------------------------
def resolve_existing_image_path(relative_path):

    # Normalize separators
    relative_path = relative_path.replace("\\", "/")

    candidates = []

    # Case 1: split path contains raw/color/
    marker = "raw/color/"

    if marker in relative_path:
        dataset_relative = relative_path.split(marker, 1)[1]
        candidates.append(
            os.path.join(
                PLANTVILLAGE_COLOR,
                dataset_relative
            )
        )

    # Case 2: path already starts with class directory
    candidates.append(
        os.path.join(
            PLANTVILLAGE_COLOR,
            relative_path
        )
    )

    # Remove leading color/ if present
    if relative_path.startswith("color/"):
        candidates.append(
            os.path.join(
                PLANTVILLAGE_COLOR,
                relative_path[len("color/"):]
            )
        )

    # Handle known whitespace-before-extension issue
    expanded_candidates = []

    for path in candidates:
        expanded_candidates.append(path)

        corrected = re.sub(
            r"\s+(\.[^.]+)$",
            r"\1",
            path
        )

        expanded_candidates.append(corrected)

    for path in expanded_candidates:
        if os.path.exists(path):
            return path

    raise FileNotFoundError(
        "Could not resolve image:\n"
        + relative_path
    )

# ------------------------------------------------------------
# Build dataframe
# ------------------------------------------------------------
def build_records(path_list):

    records = []

    for relative_path in path_list:

        relative_path = relative_path.strip()

        full_path = resolve_existing_image_path(
            relative_path
        )

        # Extract class name from the path
        class_name = os.path.basename(
            os.path.dirname(full_path)
        )

        if class_name not in CLASS_TO_IDX:
            raise ValueError(
                f"Unknown class: {class_name}"
            )

        records.append({
            "path": full_path,
            "label": CLASS_TO_IDX[class_name],
            "class_name": class_name
        })

    return pd.DataFrame(records)

# ------------------------------------------------------------
# Build official train/test
# ------------------------------------------------------------
official_train_df = build_records(
    official_train_paths
)

test_df = build_records(
    official_test_paths
)

# ------------------------------------------------------------
# Exact train / validation split
# ------------------------------------------------------------
train_df = official_train_df.iloc[
    train_idx
].reset_index(drop=True)

val_df = official_train_df.iloc[
    val_idx
].reset_index(drop=True)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------
print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))

print("\nClasses:")
print("Train      :", train_df["label"].nunique())
print("Validation :", val_df["label"].nunique())
print("Test       :", test_df["label"].nunique())

print("\nMissing paths:")
print(
    "Train      :",
    (~train_df["path"].map(os.path.exists)).sum()
)
print(
    "Validation :",
    (~val_df["path"].map(os.path.exists)).sum()
)
print(
    "Test       :",
    (~test_df["path"].map(os.path.exists)).sum()
)

assert len(train_df) == 39207
assert len(val_df) == 4389
assert len(test_df) == 10709

assert train_df["label"].nunique() == 38
assert val_df["label"].nunique() == 38
assert test_df["label"].nunique() == 38

assert train_df["path"].map(os.path.exists).all()
assert val_df["path"].map(os.path.exists).all()
assert test_df["path"].map(os.path.exists).all()

print("\nDataset reconstruction: PASS")

Train      : 39207
Validation : 4389
Test       : 10709

Classes:
Train      : 38
Validation : 38
Test       : 38

Missing paths:
Train      : 0
Validation : 0
Test       : 0

Dataset reconstruction: PASS


In [8]:
# ============================================================
# CELL 5 — Analyze Training Class Imbalance
# ============================================================

class_counts = (
    train_df["class_name"]
    .value_counts()
    .sort_index()
)

distribution_df = pd.DataFrame({
    "class_name": class_counts.index,
    "count": class_counts.values
})

distribution_df["percentage"] = (
    distribution_df["count"] / len(train_df) * 100
)

print("Total training images:", len(train_df))
print("Number of classes:", len(distribution_df))

print("\nClass distribution:")
display(distribution_df)

print("\nImbalance summary:")
print("Largest class:", distribution_df["count"].max())
print("Smallest class:", distribution_df["count"].min())
print(
    "Imbalance ratio:",
    round(
        distribution_df["count"].max() /
        distribution_df["count"].min(),
        2
    )
)

print("\nSmallest 10 classes:")
display(
    distribution_df
    .sort_values("count")
    .head(10)
)

print("\nLargest 10 classes:")
display(
    distribution_df
    .sort_values("count", ascending=False)
    .head(10)
)

Total training images: 39207
Number of classes: 38

Class distribution:


,class_name,count,percentage
0,Apple___Apple_scab,460,1.173260
1,Apple___Black_rot,430,1.096743
2,Apple___Cedar_apple_rust,203,0.517765
3,Apple___healthy,1209,3.083633
4,Blueberry___healthy,1069,2.726554
5,Cherry_(including_sour)___Powdery_mildew,710,1.810901
6,Cherry_(including_sour)___healthy,642,1.637463
7,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,369,0.941158
8,Corn_(maize)___Common_rust_,879,2.241947
9,Corn_(maize)___Northern_Leaf_Blight,687,1.752238



Imbalance summary:
Largest class: 4069
Smallest class: 108
Imbalance ratio: 37.68

Smallest 10 classes:


,class_name,count,percentage
22,Potato___healthy,108,0.275461
23,Raspberry___healthy,190,0.484607
2,Apple___Cedar_apple_rust,203,0.517765
17,Peach___healthy,243,0.619787
36,Tomato___Tomato_mosaic_virus,276,0.703956
14,Grape___healthy,294,0.749866
27,Strawberry___healthy,338,0.862091
7,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,369,0.941158
1,Apple___Black_rot,430,1.096743
0,Apple___Apple_scab,460,1.173260



Largest 10 classes:


,class_name,count,percentage
15,Orange___Haunglongbing_(Citrus_greening),4069,10.378249
35,Tomato___Tomato_Yellow_Leaf_Curl_Virus,3837,9.786518
24,Soybean___healthy,3708,9.457495
16,Peach___Bacterial_spot,1579,4.027342
28,Tomato___Bacterial_spot,1519,3.874308
30,Tomato___Late_blight,1395,3.558038
25,Squash___Powdery_mildew,1310,3.341240
32,Tomato___Septoria_leaf_spot,1300,3.315734
3,Apple___healthy,1209,3.083633
33,Tomato___Spider_mites Two-spotted_spider_mite,1208,3.081082


**Create the oversampling weights**

In [9]:
# ============================================================
# CELL 6 — Moderated Class-Balanced Oversampling Weights
# ============================================================

import numpy as np
import torch

# Class counts in the training set
class_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
    .values
)

# ------------------------------------------------------------
# Square-root inverse-frequency weighting
#
# weight_c = 1 / sqrt(class_count_c)
#
# This provides moderated oversampling without forcing
# every class to match the largest class exactly.
# ------------------------------------------------------------
class_weights = 1.0 / np.sqrt(class_counts)

# Normalize weights for easier inspection
class_weights = class_weights / class_weights.mean()

# Assign each training image the weight of its class
sample_weights = train_df["label"].map(
    dict(enumerate(class_weights))
).values.astype(np.float64)

print("Number of classes:", len(class_weights))
print("Number of samples:", len(sample_weights))

print("\nClass weight summary:")
print("Minimum:", class_weights.min())
print("Maximum:", class_weights.max())
print(
    "Weight ratio:",
    round(class_weights.max() / class_weights.min(), 2)
)

print("\nClass weights:")
for idx, (count, weight) in enumerate(
    zip(class_counts, class_weights)
):
    print(
        f"{idx:02d} | "
        f"count={count:4d} | "
        f"weight={weight:.4f}"
    )

assert len(sample_weights) == len(train_df)
assert np.isfinite(sample_weights).all()
assert (sample_weights > 0).all()

print("\nOversampling weight construction: PASS")

Number of classes: 38
Number of samples: 39207

Class weight summary:
Minimum: 0.40003445663201714
Maximum: 2.4554402337038623
Weight ratio: 6.14

Class weights:
00 | count= 460 | weight=1.1898
01 | count= 430 | weight=1.2306
02 | count= 203 | weight=1.7910
03 | count=1209 | weight=0.7339
04 | count=1069 | weight=0.7805
05 | count= 710 | weight=0.9577
06 | count= 642 | weight=1.0071
07 | count= 369 | weight=1.3284
08 | count= 879 | weight=0.8607
09 | count= 687 | weight=0.9736
10 | count= 821 | weight=0.8906
11 | count= 916 | weight=0.8431
12 | count= 987 | weight=0.8122
13 | count= 788 | weight=0.9090
14 | count= 294 | weight=1.4882
15 | count=4069 | weight=0.4000
16 | count=1579 | weight=0.6422
17 | count= 243 | weight=1.6370
18 | count= 756 | weight=0.9281
19 | count=1104 | weight=0.7680
20 | count= 748 | weight=0.9330
21 | count= 712 | weight=0.9563
22 | count= 108 | weight=2.4554
23 | count= 190 | weight=1.8512
24 | count=3708 | weight=0.4191
25 | count=1310 | weight=0.7050
26 | c

In [10]:
# ============================================================
# CELL 7 — Verify Expected Oversampled Distribution
# ============================================================

# Sampling probability for each training image
sample_probabilities = (
    sample_weights / sample_weights.sum()
)

# Expected number of samples per class in one epoch
expected_class_samples = np.zeros(len(CLASS_NAMES))

for class_idx in range(len(CLASS_NAMES)):
    class_mask = train_df["label"].values == class_idx
    expected_class_samples[class_idx] = (
        sample_probabilities[class_mask].sum()
        * len(train_df)
    )

expected_df = pd.DataFrame({
    "class_name": CLASS_NAMES,
    "original_count": class_counts,
    "sampling_weight": class_weights,
    "expected_samples_per_epoch": expected_class_samples
})

expected_df["expected_percentage"] = (
    expected_df["expected_samples_per_epoch"]
    / len(train_df)
    * 100
)

print("Expected samples per epoch:", len(train_df))

print("\nSmallest expected classes:")
display(
    expected_df
    .sort_values("expected_samples_per_epoch")
    .head(10)
)

print("\nLargest expected classes:")
display(
    expected_df
    .sort_values(
        "expected_samples_per_epoch",
        ascending=False
    )
    .head(10)
)

print("\nExpected distribution summary:")
print(
    "Minimum expected:",
    round(expected_df["expected_samples_per_epoch"].min(), 1)
)
print(
    "Maximum expected:",
    round(expected_df["expected_samples_per_epoch"].max(), 1)
)

print(
    "Expected ratio:",
    round(
        expected_df["expected_samples_per_epoch"].max()
        / expected_df["expected_samples_per_epoch"].min(),
        2
    )
)

assert np.isclose(
    expected_df["expected_samples_per_epoch"].sum(),
    len(train_df)
)

print("\nExpected sampling distribution: PASS")

Expected samples per epoch: 39207

Smallest expected classes:


,class_name,original_count,sampling_weight,expected_samples_per_epoch,expected_percentage
22,Potato___healthy,108,2.455440,359.826745,0.917761
23,Raspberry___healthy,190,1.851247,477.263655,1.217292
2,Apple___Cedar_apple_rust,203,1.790990,493.320975,1.258247
17,Peach___healthy,243,1.636960,539.740117,1.376642
36,Tomato___Tomato_mosaic_virus,276,1.535984,575.222815,1.467143
14,Grape___healthy,294,1.488222,593.683840,1.514229
27,Strawberry___healthy,338,1.387979,636.560832,1.623590
7,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,369,1.328398,665.111942,1.696411
1,Apple___Black_rot,430,1.230572,717.985693,1.831269
0,Apple___Apple_scab,460,1.189768,742.609461,1.894074



Largest expected classes:


,class_name,original_count,sampling_weight,expected_samples_per_epoch,expected_percentage
15,Orange___Haunglongbing_(Citrus_greening),4069,0.400034,2208.642410,5.633286
35,Tomato___Tomato_Yellow_Leaf_Curl_Virus,3837,0.411951,2144.753879,5.470334
24,Soybean___healthy,3708,0.419055,2108.392317,5.377592
16,Peach___Bacterial_spot,1579,0.642170,1375.854876,3.509207
28,Tomato___Bacterial_spot,1519,0.654730,1349.461346,3.441889
30,Tomato___Late_blight,1395,0.683210,1293.208844,3.298413
25,Squash___Powdery_mildew,1310,0.705027,1253.190835,3.196345
32,Tomato___Septoria_leaf_spot,1300,0.707733,1248.398501,3.184121
3,Apple___healthy,1209,0.733885,1203.911915,3.070656
33,Tomato___Spider_mites Two-spotted_spider_mite,1208,0.734189,1203.413916,3.069385



Expected distribution summary:
Minimum expected: 359.8
Maximum expected: 2208.6
Expected ratio: 6.14

Expected sampling distribution: PASS


In [11]:
# ============================================================
# CELL 8 — Create WeightedRandomSampler
# ============================================================

from torch.utils.data import WeightedRandomSampler

# Convert sample weights to a PyTorch tensor
sample_weights_tensor = torch.as_tensor(
    sample_weights,
    dtype=torch.double
)

# Create weighted sampler
oversampling_sampler = WeightedRandomSampler(
    weights=sample_weights_tensor,
    num_samples=len(train_df),
    replacement=True
)

print("Sampler type:", type(oversampling_sampler).__name__)
print("Samples per epoch:", len(oversampling_sampler))
print("Replacement sampling:", True)

assert len(oversampling_sampler) == len(train_df)

print("\nWeightedRandomSampler creation: PASS")

Sampler type: WeightedRandomSampler
Samples per epoch: 39207
Replacement sampling: True

WeightedRandomSampler creation: PASS


In [12]:
# ============================================================
# CELL 9 — Verify Actual Weighted Sampling Distribution
# ============================================================

# Draw one complete sampled epoch
sampled_indices = list(iter(oversampling_sampler))

sampled_labels = train_df.iloc[sampled_indices]["label"].values

actual_sampled_counts = np.bincount(
    sampled_labels,
    minlength=len(CLASS_NAMES)
)

sampling_check_df = pd.DataFrame({
    "class_name": CLASS_NAMES,
    "original_count": class_counts,
    "expected_samples": np.round(
        expected_class_samples
    ).astype(int),
    "actual_samples": actual_sampled_counts
})

sampling_check_df["difference"] = (
    sampling_check_df["actual_samples"]
    - sampling_check_df["expected_samples"]
)

sampling_check_df["actual_percentage"] = (
    sampling_check_df["actual_samples"]
    / len(sampled_indices)
    * 100
)

print("Sampled images:", len(sampled_indices))
print("Unique images sampled:", len(set(sampled_indices)))

print("\nActual vs expected distribution:")
display(sampling_check_df)

print("\nDistribution summary:")
print(
    "Minimum actual:",
    actual_sampled_counts.min()
)
print(
    "Maximum actual:",
    actual_sampled_counts.max()
)

print(
    "Actual sampling ratio:",
    round(
        actual_sampled_counts.max()
        / actual_sampled_counts.min(),
        2
    )
)

print("\nOriginal vs sampled totals:")
print(
    "Original training images:",
    len(train_df)
)
print(
    "Sampled training images:",
    len(sampled_indices)
)

assert len(sampled_indices) == len(train_df)
assert actual_sampled_counts.sum() == len(train_df)

print("\nActual sampler verification: PASS")

Sampled images: 39207
Unique images sampled: 23666

Actual vs expected distribution:


,class_name,original_count,expected_samples,actual_samples,difference,actual_percentage
0,Apple___Apple_scab,460,743,699,-44,1.782845
1,Apple___Black_rot,430,718,745,27,1.900171
2,Apple___Cedar_apple_rust,203,493,466,-27,1.188563
3,Apple___healthy,1209,1204,1179,-25,3.007116
4,Blueberry___healthy,1069,1132,1150,18,2.933150
5,Cherry_(including_sour)___Powdery_mildew,710,923,980,57,2.499554
6,Cherry_(including_sour)___healthy,642,877,862,-15,2.198587
7,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,369,665,712,47,1.816002
8,Corn_(maize)___Common_rust_,879,1027,1036,9,2.642385
9,Corn_(maize)___Northern_Leaf_Blight,687,908,871,-37,2.221542



Distribution summary:
Minimum actual: 355
Maximum actual: 2228
Actual sampling ratio: 6.28

Original vs sampled totals:
Original training images: 39207
Sampled training images: 39207

Actual sampler verification: PASS


In [13]:
# ============================================================
# CELL 10 — Standard A0 Transforms
# ============================================================

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMAGE_SIZE = 224
BATCH_SIZE = 32

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Same standard training augmentation used in A0
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])

# Validation/test: no augmentation
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])

class PlantVillageDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(
            row["path"]
        ).convert("RGB")

        label = int(row["label"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label

# Create datasets
oversampling_train_dataset = PlantVillageDataset(
    train_df,
    transform=train_transform
)

val_dataset = PlantVillageDataset(
    val_df,
    transform=eval_transform
)

test_dataset = PlantVillageDataset(
    test_df,
    transform=eval_transform
)

print("Train dataset:", len(oversampling_train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

print("\nTransform setup: PASS")

Train dataset: 39207
Validation dataset: 4389
Test dataset: 10709

Transform setup: PASS


In [14]:
# ============================================================
# CELL 11 — Oversampled Train Loader + Normal Eval Loaders
# ============================================================

from torch.utils.data import DataLoader

# Oversampled training loader
oversampling_train_loader = DataLoader(
    oversampling_train_dataset,
    batch_size=BATCH_SIZE,
    sampler=oversampling_sampler,
    num_workers=2,
    pin_memory=True
)

# Validation loader — unchanged
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Test loader — unchanged
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------------------
# Verify one training batch
# ------------------------------------------------------------
images, labels = next(iter(oversampling_train_loader))

print("Training batch images:", images.shape)
print("Training batch labels :", labels.shape)
print("Image dtype           :", images.dtype)
print("Label dtype           :", labels.dtype)
print(
    "Label range           :",
    int(labels.min()),
    "to",
    int(labels.max())
)

print("\nValidation batches:", len(val_loader))
print("Test batches:", len(test_loader))

assert images.shape == (BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE)
assert labels.shape == (BATCH_SIZE,)
assert labels.dtype == torch.int64

print("\nDataLoader setup: PASS")

Training batch images: torch.Size([32, 3, 224, 224])
Training batch labels : torch.Size([32])
Image dtype           : torch.float32
Label dtype           : torch.int64
Label range           : 3 to 36

Validation batches: 138
Test batches: 335

DataLoader setup: PASS


In [15]:
# ============================================================
# CELL 12 — ResNet50 Model Setup
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

SEED = 42
NUM_CLASSES = 38
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Reproducibility
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Pretrained ResNet50 — same as corrected A0 protocol
# ------------------------------------------------------------
model = models.resnet50(
    weights=models.ResNet50_Weights.IMAGENET1K_V2
)

# Replace classifier
model.fc = nn.Linear(
    model.fc.in_features,
    NUM_CLASSES
)

# Freeze backbone
for parameter in model.parameters():
    parameter.requires_grad = False

# Train only final classifier
for parameter in model.fc.parameters():
    parameter.requires_grad = True

model = model.to(DEVICE)

# ------------------------------------------------------------
# Parameter verification
# ------------------------------------------------------------
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

# Forward-pass verification
dummy_input = torch.randn(
    2, 3, IMAGE_SIZE, IMAGE_SIZE,
    device=DEVICE
)

with torch.no_grad():
    dummy_output = model(dummy_input)

print("Device:", DEVICE)
print("Classes:", NUM_CLASSES)
print("Input size:", IMAGE_SIZE)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

print(
    "Forward:",
    tuple(dummy_input.shape),
    "->",
    tuple(dummy_output.shape)
)

assert total_params == 23_585_894
assert trainable_params == 77_862
assert dummy_output.shape == (2, NUM_CLASSES)

print("\nResNet50 model verification: PASS")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 217MB/s]


Device: cuda
Classes: 38
Input size: 224
Total parameters: 23585894
Trainable parameters: 77862
Forward: (2, 3, 224, 224) -> (2, 38)

ResNet50 model verification: PASS


In [16]:
# ============================================================
# CELL 13 — Training Configuration
# ============================================================

import torch.optim as optim

LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
MAX_EPOCHS = 10
PATIENCE = 3

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Training configuration:")
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Loss:", criterion.__class__.__name__)
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", MAX_EPOCHS)
print("Early stopping patience:", PATIENCE)

print("\nOnly trainable parameters will be optimized:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

assert LEARNING_RATE == 0.001
assert WEIGHT_DECAY == 0.0001
assert trainable_params == 77862

print("\nTraining configuration: PASS")

Training configuration:
Optimizer: AdamW
Learning rate: 0.001
Weight decay: 0.0001
Loss: CrossEntropyLoss
Batch size: 32
Maximum epochs: 10
Early stopping patience: 3

Only trainable parameters will be optimized: 77862

Training configuration: PASS


In [17]:
# ============================================================
# CELL 14 — Training & Evaluation Functions
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    running_loss = 0.0
    all_labels = []
    all_predictions = []

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()
        optimizer.step()

        running_loss += (
            loss.item() * images.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

    epoch_loss = (
        running_loss / len(loader.dataset)
    )

    epoch_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    epoch_macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return (
        epoch_loss,
        epoch_accuracy,
        epoch_macro_f1
    )


@torch.no_grad()
def evaluate_model(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    running_loss = 0.0
    all_labels = []
    all_predictions = []

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        running_loss += (
            loss.item() * images.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

    epoch_loss = (
        running_loss / len(loader.dataset)
    )

    epoch_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    epoch_macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    epoch_weighted_f1 = f1_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    epoch_macro_precision = precision_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    epoch_macro_recall = recall_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return {
        "loss": epoch_loss,
        "accuracy": epoch_accuracy,
        "macro_f1": epoch_macro_f1,
        "weighted_f1": epoch_weighted_f1,
        "macro_precision": epoch_macro_precision,
        "macro_recall": epoch_macro_recall,
        "labels": all_labels,
        "predictions": all_predictions
    }


print("Training function: defined")
print("Evaluation function: defined")

# Quick sanity check
assert callable(train_one_epoch)
assert callable(evaluate_model)

print("\nTraining/evaluation functions: PASS")

Training function: defined
Evaluation function: defined

Training/evaluation functions: PASS


In [18]:
# ============================================================
# CELL 15 — Pre-training Sanity Check
# ============================================================

# Check one training batch
train_images, train_labels = next(
    iter(oversampling_train_loader)
)

train_images = train_images.to(
    DEVICE,
    non_blocking=True
)

train_labels = train_labels.to(
    DEVICE,
    non_blocking=True
)

model.train()

optimizer.zero_grad()

train_outputs = model(train_images)
train_loss = criterion(
    train_outputs,
    train_labels
)

train_loss.backward()

print("Training batch:")
print("Images :", tuple(train_images.shape))
print("Labels :", tuple(train_labels.shape))
print("Output :", tuple(train_outputs.shape))
print("Loss   :", round(train_loss.item(), 6))

# Check that classifier received gradients
fc_grad_exists = (
    model.fc.weight.grad is not None
)

print(
    "Classifier gradient exists:",
    fc_grad_exists
)

# Check frozen backbone has no gradient
backbone_grad_exists = any(
    p.grad is not None
    for name, p in model.named_parameters()
    if not name.startswith("fc.")
)

print(
    "Frozen backbone gradient exists:",
    backbone_grad_exists
)

assert train_outputs.shape == (
    BATCH_SIZE,
    NUM_CLASSES
)

assert train_loss.ndim == 0
assert fc_grad_exists
assert not backbone_grad_exists

# Clear gradients before real training
optimizer.zero_grad(set_to_none=True)

print("\nPre-training sanity check: PASS")

Training batch:
Images : (32, 3, 224, 224)
Labels : (32,)
Output : (32, 38)
Loss   : 3.656631
Classifier gradient exists: True
Frozen backbone gradient exists: False

Pre-training sanity check: PASS


In [19]:
# ============================================================
# CELL 16 — ResNet50 + Oversampling Training
#           CHECKPOINT PROTECTED
# ============================================================

import os
import copy
import time
import json
import pandas as pd
import torch

# ------------------------------------------------------------
# Checkpoint directory
# ------------------------------------------------------------
CHECKPOINT_DIR = "/kaggle/working/resnet50_oversampling_checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "best_model.pth"
)

LATEST_CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "latest_checkpoint.pth"
)

HISTORY_PATH = os.path.join(
    CHECKPOINT_DIR,
    "training_history.csv"
)

STATE_PATH = os.path.join(
    CHECKPOINT_DIR,
    "training_state.json"
)

print("Checkpoint directory:")
print(CHECKPOINT_DIR)

# ------------------------------------------------------------
# Training state
# ------------------------------------------------------------
history = []

best_val_macro_f1 = -1.0
best_epoch = 0
best_state_dict = None
early_stop_counter = 0

# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------
for epoch in range(1, MAX_EPOCHS + 1):

    epoch_start = time.time()

    # ========================================================
    # TRAINING
    # ========================================================
    train_loss, train_acc, train_macro_f1 = train_one_epoch(
        model,
        oversampling_train_loader,
        criterion,
        optimizer,
        DEVICE
    )

    # ========================================================
    # VALIDATION
    # ========================================================
    val_results = evaluate_model(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    epoch_time = time.time() - epoch_start

    # ========================================================
    # RECORD HISTORY
    # ========================================================
    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_macro_f1": train_macro_f1,
        "val_loss": val_results["loss"],
        "val_accuracy": val_results["accuracy"],
        "val_macro_f1": val_results["macro_f1"],
        "val_weighted_f1": val_results["weighted_f1"],
        "val_macro_precision": val_results["macro_precision"],
        "val_macro_recall": val_results["macro_recall"],
        "epoch_time_seconds": epoch_time
    }

    history.append(epoch_record)

    # ========================================================
    # CHECK WHETHER THIS IS THE BEST MODEL
    # ========================================================
    is_best = (
        val_results["macro_f1"] >
        best_val_macro_f1
    )

    if is_best:

        best_val_macro_f1 = val_results["macro_f1"]
        best_epoch = epoch
        best_state_dict = copy.deepcopy(
            model.state_dict()
        )

        early_stop_counter = 0

    else:

        early_stop_counter += 1

    # ========================================================
    # SAVE LATEST CHECKPOINT — EVERY EPOCH
    # ========================================================
    checkpoint = {
        "epoch": epoch,

        "model_state_dict": model.state_dict(),

        "optimizer_state_dict": optimizer.state_dict(),

        "history": history,

        "best_val_macro_f1": best_val_macro_f1,

        "best_epoch": best_epoch,

        "early_stop_counter": early_stop_counter,

        "experiment_config": {
            "experiment": "ResNet50 + Moderated Oversampling",
            "dataset": "PlantVillage",
            "num_classes": NUM_CLASSES,
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "optimizer": "AdamW",
            "loss": "CrossEntropyLoss",
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "seed": SEED,
            "sampling_strategy": "sqrt_inverse_frequency",
            "sampling_weight_ratio": float(
                class_weights.max() /
                class_weights.min()
            ),
            "train_samples": len(train_df),
            "validation_samples": len(val_df),
            "test_samples": len(test_df)
        }
    }

    torch.save(
        checkpoint,
        LATEST_CHECKPOINT_PATH
    )

    # ========================================================
    # SAVE HISTORY — EVERY EPOCH
    # ========================================================
    pd.DataFrame(history).to_csv(
        HISTORY_PATH,
        index=False
    )

    # ========================================================
    # SAVE TRAINING STATE — EVERY EPOCH
    # ========================================================
    state = {
        "last_completed_epoch": epoch,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "early_stop_counter": early_stop_counter,
        "epochs_completed": len(history),
        "latest_checkpoint": LATEST_CHECKPOINT_PATH,
        "best_checkpoint": BEST_CHECKPOINT_PATH
    }

    with open(STATE_PATH, "w") as f:
        json.dump(
            state,
            f,
            indent=2
        )

    # ========================================================
    # SAVE BEST MODEL
    # ========================================================
    if is_best:

        best_checkpoint = {
            "epoch": epoch,

            "model_state_dict":
                copy.deepcopy(
                    model.state_dict()
                ),

            "optimizer_state_dict":
                copy.deepcopy(
                    optimizer.state_dict()
                ),

            "history":
                copy.deepcopy(history),

            "best_val_macro_f1":
                best_val_macro_f1,

            "best_epoch":
                best_epoch,

            "early_stop_counter":
                early_stop_counter,

            "experiment_config":
                checkpoint["experiment_config"]
        }

        torch.save(
            best_checkpoint,
            BEST_CHECKPOINT_PATH
        )

    # ========================================================
    # PRINT RESULTS
    # ========================================================
    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Train Acc: {train_acc:.6f} | "
        f"Train Macro F1: {train_macro_f1:.6f} | "
        f"Val Loss: {val_results['loss']:.6f} | "
        f"Val Acc: {val_results['accuracy']:.6f} | "
        f"Val Macro F1: {val_results['macro_f1']:.6f} | "
        f"Time: {epoch_time:.1f}s"
    )

    if is_best:

        print(
            f"  -> NEW BEST "
            f"(Val Macro F1: "
            f"{best_val_macro_f1:.6f})"
        )

        print(
            f"  -> Best checkpoint saved: "
            f"{BEST_CHECKPOINT_PATH}"
        )

    else:

        print(
            f"  -> No improvement "
            f"({early_stop_counter}/{PATIENCE})"
        )

    print(
        f"  -> Latest checkpoint saved: "
        f"{LATEST_CHECKPOINT_PATH}"
    )

    # ========================================================
    # EARLY STOPPING
    # ========================================================
    if early_stop_counter >= PATIENCE:

        print(
            f"\nEarly stopping triggered "
            f"at epoch {epoch}."
        )

        break

# ------------------------------------------------------------
# Restore best model
# ------------------------------------------------------------
best_checkpoint_loaded = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    best_checkpoint_loaded["model_state_dict"]
)

model.eval()

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print("Best epoch:", best_checkpoint_loaded["best_epoch"])

print(
    "Best validation Macro F1:",
    f"{best_checkpoint_loaded['best_val_macro_f1']:.6f}"
)

print(
    "Epochs completed:",
    len(best_checkpoint_loaded["history"])
)

print("\nSaved files:")

for path in [
    BEST_CHECKPOINT_PATH,
    LATEST_CHECKPOINT_PATH,
    HISTORY_PATH,
    STATE_PATH
]:

    print(
        os.path.basename(path),
        "|",
        round(os.path.getsize(path) / (1024 ** 2), 2),
        "MB"
    )

print("\nBest model restored successfully.")

Checkpoint directory:
/kaggle/working/resnet50_oversampling_checkpoints
Epoch 01 | Train Loss: 0.753564 | Train Acc: 0.859081 | Train Macro F1: 0.858177 | Val Loss: 0.318912 | Val Acc: 0.933242 | Val Macro F1: 0.915768 | Time: 244.5s
  -> NEW BEST (Val Macro F1: 0.915768)
  -> Best checkpoint saved: /kaggle/working/resnet50_oversampling_checkpoints/best_model.pth
  -> Latest checkpoint saved: /kaggle/working/resnet50_oversampling_checkpoints/latest_checkpoint.pth
Epoch 02 | Train Loss: 0.252985 | Train Acc: 0.941771 | Train Macro F1: 0.940672 | Val Loss: 0.195449 | Val Acc: 0.951242 | Val Macro F1: 0.939735 | Time: 218.2s
  -> NEW BEST (Val Macro F1: 0.939735)
  -> Best checkpoint saved: /kaggle/working/resnet50_oversampling_checkpoints/best_model.pth
  -> Latest checkpoint saved: /kaggle/working/resnet50_oversampling_checkpoints/latest_checkpoint.pth
Epoch 03 | Train Loss: 0.182326 | Train Acc: 0.954141 | Train Macro F1: 0.953696 | Val Loss: 0.149145 | Val Acc: 0.964001 | Val Macro F1

In [20]:
# ============================================================
# CELL 17 — Clean Test Evaluation
# ============================================================

# Best model is already restored after Cell 16
model.eval()

test_results = evaluate_model(
    model,
    test_loader,
    criterion,
    DEVICE
)

print("=" * 70)
print("RESNET50 + OVERSAMPLING — CLEAN TEST RESULTS")
print("=" * 70)

print(
    f"Test Loss       : {test_results['loss']:.6f}"
)

print(
    f"Accuracy        : {test_results['accuracy']:.6f} "
    f"({test_results['accuracy'] * 100:.2f}%)"
)

print(
    f"Macro F1        : {test_results['macro_f1']:.6f} "
    f"({test_results['macro_f1'] * 100:.2f}%)"
)

print(
    f"Weighted F1     : {test_results['weighted_f1']:.6f} "
    f"({test_results['weighted_f1'] * 100:.2f}%)"
)

print(
    f"Macro Precision  : {test_results['macro_precision']:.6f} "
    f"({test_results['macro_precision'] * 100:.2f}%)"
)

print(
    f"Macro Recall     : {test_results['macro_recall']:.6f} "
    f"({test_results['macro_recall'] * 100:.2f}%)"
)

correct = sum(
    np.array(test_results["labels"]) ==
    np.array(test_results["predictions"])
)

incorrect = len(test_results["labels"]) - correct

print("Correct         :", correct)
print("Incorrect       :", incorrect)

assert len(test_results["labels"]) == 10709
assert len(test_results["predictions"]) == 10709

print("\nClean test evaluation: PASS")

RESNET50 + OVERSAMPLING — CLEAN TEST RESULTS
Test Loss       : 0.096439
Accuracy        : 0.971146 (97.11%)
Macro F1        : 0.962715 (96.27%)
Weighted F1     : 0.971048 (97.10%)
Macro Precision  : 0.961686 (96.17%)
Macro Recall     : 0.964353 (96.44%)
Correct         : 10400
Incorrect       : 309

Clean test evaluation: PASS


In [29]:
# CELL 19 — Paper-Ready Oversampling Comparison Table

import pandas as pd

# ------------------------------------------------------------
# Exact results from the completed experiments
# ------------------------------------------------------------

a0_results = {
    "Model": "ResNet50 (A0)",
    "Accuracy (%)": 96.7597,
    "Macro F1 (%)": 95.7727,
    "Weighted F1 (%)": 96.7431,
    "Macro Precision (%)": 95.9533,
    "Macro Recall (%)": 95.7432
}

oversampling_results = {
    "Model": "ResNet50 + Oversampling",
    "Accuracy (%)": 97.1146,
    "Macro F1 (%)": 96.2715,
    "Weighted F1 (%)": 97.1048,
    "Macro Precision (%)": 96.1686,
    "Macro Recall (%)": 96.4353
}

improvement = {
    "Model": "Improvement (pp)",
    "Accuracy (%)": 97.1146 - 96.7597,
    "Macro F1 (%)": 96.2715 - 95.7727,
    "Weighted F1 (%)": 97.1048 - 96.7431,
    "Macro Precision (%)": 96.1686 - 95.9533,
    "Macro Recall (%)": 96.4353 - 95.7432
}

# ------------------------------------------------------------
# Create table
# ------------------------------------------------------------

oversampling_table = pd.DataFrame([
    a0_results,
    oversampling_results,
    improvement
])

# Round for paper presentation
oversampling_table_display = oversampling_table.copy()

metric_columns = [
    "Accuracy (%)",
    "Macro F1 (%)",
    "Weighted F1 (%)",
    "Macro Precision (%)",
    "Macro Recall (%)"
]

oversampling_table_display[metric_columns] = (
    oversampling_table_display[metric_columns].round(2)
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 100)
print("PAPER-READY TABLE — EFFECT OF OVERSAMPLING")
print("=" * 100)

display(oversampling_table_display)

# ------------------------------------------------------------
# Save CSV
# ------------------------------------------------------------

output_path = (
    "/kaggle/working/"
    "oversampling_comparison_paper_table.csv"
)

oversampling_table_display.to_csv(
    output_path,
    index=False
)

print(f"\nSaved to:")
print(output_path)

print("\nTable title:")
print(
    "Table X. Effect of class-balanced oversampling "
    "on ResNet50 performance."
)

PAPER-READY TABLE — EFFECT OF OVERSAMPLING


,Model,Accuracy (%),Macro F1 (%),Weighted F1 (%),Macro Precision (%),Macro Recall (%)
0,ResNet50 (A0),96.76,95.77,96.74,95.95,95.74
1,ResNet50 + Oversampling,97.11,96.27,97.10,96.17,96.44
2,Improvement (pp),0.35,0.50,0.36,0.22,0.69



Saved to:
/kaggle/working/oversampling_comparison_paper_table.csv

Table title:
Table X. Effect of class-balanced oversampling on ResNet50 performance.
